In [ ]:
!pip install ortools

In [2]:
from ortools.linear_solver import pywraplp
import pandas as pd

In [6]:
df_dados_gerais = pd.read_csv('/content/sample_data/dados-gerais.csv')
df_dados_arcos = pd.read_csv('/content/sample_data/dados-arcos.csv')

In [26]:
num_vertices = df_dados_gerais['num_vertices'][0]
origem_caminho = df_dados_gerais['origem_caminho'][0]
destino_caminho = df_dados_gerais['destino_caminho'][0]

vertices = []
for i in range(1, num_vertices+1):
  vertices.append(i)

arcos = []
for row in df_dados_arcos.itertuples():
  arcos.append((row.origem, row.destino, row.custo))

In [43]:
solver = pywraplp.Solver.CreateSolver('SCIP')
infinity = solver.infinity()

In [44]:
x = {}
for a in arcos:
  i, j = a[0], a[1]
  x[(i,j)] = solver.BoolVar(f'x{i}{j}')

In [45]:
objetivo = solver.Objective()
for a in arcos:
  i, j, c = a[0], a[1], a[2]
  objetivo.SetCoefficient(x[(i,j)], c)
objetivo.SetMinimization()

In [53]:
for v in vertices:
  restricao = None
  if v == origem_caminho:
    restricao = solver.Constraint(1, 1, f'vertice_{v}')
  elif v == destino_caminho:
    restricao = solver.Constraint(-1, -1, f'vertice_{v}')
  else:
    restricao = solver.Constraint(0, 0, f'vertice_{v}')
  for a in arcos:
    if a[0] == v:
      i, j = a[0], a[1]
      restricao.SetCoefficient(x[(i,j)], 1)
    if a[1] == v:
      i, j = a[0], a[1]
      restricao.SetCoefficient(x[(i,j)], -1)

In [ ]:
status = solver.Solve() #resolver o modelo # x[(2,3)].SetBounds(0,0)
 

In [ ]:
if status == pywraplp.Solver.OPTIMAL:
  print('Solução:')
  print(f'Custo total: {objetivo.Value()}')
  for a in arcos:
    i, j = a[0], a[1]
    if x[(i,j)].solution_value() > 0.5:
      print(f'Arco {i} -> {j}')
else:
  print('O problema não tem solução ótima.')

In [ ]:
print(solver.ExportModelAsLpFormat(False))